In [1]:
import numpy as np
import sys 
import sys
sys.path.append('../scripts')
import assembly_tree as at
sys.path.append('../scripts')
import mcmc 
import networkx as nx 
import matplotlib.pyplot as plt 
import numpy as np 
import json
import os
import treelib
import importlib
import copy
#import pandas as pd

importlib.reload(at)

<module 'assembly_tree' from '/Users/glover.co/Documents/laszlo/NetDesign/scripts/assembly_tree.py'>

In [2]:
# Open json
with open('../data/proteins/human/treefiles/CPX-1919_tree.json','r') as f:
    tree_data = json.load(f)

In [21]:
target = nx.read_edgelist('../data/proteins/human/edgefiles/CPX-1919.edge', nodetype=int,create_using=nx.Graph, data=(('weight', float),))
X = np.loadtxt('../data/proteins/human/Xfiles/X_CPX-1919.txt', delimiter=' ')
capacity = at.extract_deg_cap(target, X).T[0]
O = at.extract_O(target,X)

In [25]:
tree_data[0]

{'0': {'children': [{'20': {'children': [{'87': {'data': [3]}},
      {'117': {'children': [{'119': {'data': [2, 7, 13]}},
         {'120': {'data': [6, 17]}}],
        'data': [2, 6, 7, 13, 17]}}],
     'data': [2, 3, 6, 7, 13, 17]}},
   {'36': {'children': [{'105': {'data': [1]}},
      {'116': {'children': [{'106': {'data': [5, 15, 16]}},
         {'107': {'data': [8, 9]}}],
        'data': [5, 8, 9, 15, 16]}}],
     'data': [1, 5, 8, 9, 15, 16]}},
   {'118': {'children': [{'68': {'data': [4, 10]}},
      {'115': {'children': [{'121': {'data': [0, 12, 14]}},
         {'122': {'data': [11]}}],
        'data': [0, 11, 12, 14]}}],
     'data': [0, 4, 10, 11, 12, 14]}}],
  'data': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]},
 'success': 1}

In [83]:
new_tree = mcmc.AssemblyTree(target, X, O, capacity)

In [84]:
def create_tree_from_json(f,tree,parent=None):
    nodes_to_add = list(f.keys())
    if 'success' in nodes_to_add:
        nodes_to_add.remove('success')        
    while len(nodes_to_add) > 0:
        node = nodes_to_add.pop(0)
        if parent is not None:
            tree.Tree.create_node(int(node), int(node), parent=parent, data=mcmc.AssemblyNode(f[node]['data'],tree.X,tree.O,tree.capacity))
        # Check whether node has children
        if 'children' in f[str(node)]:
            children = f[str(node)]['children']
            for child in children:
                create_tree_from_json(child, tree, parent=int(node))
        tree.Tree.show()
        print(tree.Tree.get_node(node))
        new_tree.update_prob(node)
    return 

In [85]:
create_tree_from_json(tree_data[0],new_tree)

0
└── 20
    └── 87



AttributeError: 'NoneType' object has no attribute 'data'

In [86]:
new_tree.Tree.get_node(87).data

In [89]:
new_tree.update_prob(87)

In [91]:
new_tree.Tree.get_node(87).data.p

[np.float64(1.0)]